# 音频数字化

教师备课与试跑视图，含教师说明；课堂优先投影学生观察版。hide-input标签不保证JupyterLab自动隐藏代码，请课前实际检查。

In [ ]:
from pathlib import Path
import sys
# 从课程目录或其demos目录启动均可；不使用当前用户的绝对路径。
ROOT = Path.cwd()
if not (ROOT / 'demos' / 'audio_core.py').exists():
    if (ROOT.parent / 'demos' / 'audio_core.py').exists(): ROOT = ROOT.parent
    else: raise RuntimeError('请从1-2-3-audio-encoding课程目录打开Notebook')
sys.path.insert(0, str(ROOT / 'demos'))
from audio_core import plot_sampling, plot_quantization, show_audio, pcm_info, show_pcm_evidence, quantize, codes
from IPython.display import display, IFrame


## Q1 同一段音乐，为什么大小不同？

同一段6秒音乐，两个文件均未做音频压缩。
A：529,244B；B：96,044B。
先试听，再猜测：为什么大小不同？

预测：________

观察后解释：________

**教师说明（不投影）**

上节课，一只熊猫变成了像素和码字。今天先看两份真实文件：都六秒，都没有做音频压缩，大小却差很多。先听A，再听B。听出了区别或暂时没听出，都可以提出猜测。什么记录参数可能不同？我先把两条猜测留在这里，等会儿读实际文件来核对。

追问：同样是6秒，电脑在这6秒中记录了多少个数？

技术结论：音乐数字化需要记录变化；时长相同不代表数据量相同。

### 开场试听：A、B；先听，暂不揭示参数。

In [ ]:
from IPython.display import HTML
display(HTML('<p>A</p>'))
show_audio('music-44100-16-mono.wav')
display(HTML('<p>B</p>'))
show_audio('music-8000-16-mono.wav')

## Q2 声音怎样变成一条波形？

看同一位置随时间变化的声音。
波形的横轴、纵轴分别表示什么？

预测：________

观察后解释：________

**教师说明（不投影）**

先看这一条曲线。往右是时间过去，不是声音在教室里走过的距离。麦克风把声压变化转成电信号，上下表示信号相对基准的数值。谁能指一个零线下方的时刻？负值表示低于基准，不是负的音量，也不是物体朝下移动。这里画的是归一化模型，不是原音乐的实测波形。

追问：曲线在零线下方时，是否表示“负的音量”？横坐标不同，记录的是不同位置，还是不同时刻？

技术结论：音频可以用随时间变化的信号表示。图上画的是模型，不是空气中可见的一条线。

## Q3 怎样留下波形随时间的变化？

这一秒的曲线怎样记录？
在纸上圈出你准备读取的时刻。

预测：________

观察后解释：________

**教师说明（不投影）**

图像课挑位置，声音课先挑时刻。先圈几个时刻，让同桌说说怎样知道它们各在什么时候。如果每次间隔相同，就不必逐点交代时间，只要约定开始时间和每秒读取次数。再看红点：每个点留下当时的幅度，紫线是比较用的模型。点之间的整条曲线没有逐点存进去；现在先决定什么时候读，不急着决定保留几位数。

追问：随意挑点，除了幅度还要交代什么？等间隔读点时，可否只约定开始时刻和采样率？两个时刻之间的全部变化都直接写进去了吗？

技术结论：采样在离散时刻取得幅度样本。

### D1 先预测，后运行

In [ ]:
plot_sampling(12, False)

**等待讨论。下面单元只在完成预测后运行。**

In [ ]:
plot_sampling(12, True)
plot_sampling(24, True)

## Q4 每秒采样8000次，是什么意思？

单声道：每秒等间隔读取8000个样本。
采样率是多少？相邻时刻间隔多久？

预测：________

观察后解释：________

**教师说明（不投影）**

每秒读八千次，这个读数的频率叫采样率。请注意单位：八千赫兹，也可以写八千次每秒。相邻两次相隔多少时间？一秒分成八千份。声音自身的频率是在数振动，采样率是在数记录，两件事要分开。

追问：声音本身6000Hz、记录采样率24000Hz，两个数分别数什么？

技术结论：`fs`确定样本的时间间隔；kHz按1000换算。

## Q5 样本值不在可选等级上，怎么办？

四个样本：−0.62、−0.10、0.38、0.84。
只能用：−0.75、−0.25、0.25、0.75近似。
分别选哪个？

预测：________

观察后解释：________

**教师说明（不投影）**

这四个值都读到了，可是现在只允许四个等级。请先各自选最接近的一档。哪一项完全没有误差？都没有。图像课用有限颜色近似颜色，这里用有限幅度等级近似幅度，叫量化。误差来自这一档的近似，不是时刻有没有选好。

追问：−0.62与−0.75差了多少？把采样时刻加密能直接消除这次近似误差吗？

技术结论：采样与量化解决不同限制；量化通常带来误差。

## Q6 四个量化等级，怎样写成码字？

从低到高编号0、1、2、3；采用等长二进制码。
每个样本至少几bit？写出四个码字。

预测：________

观察后解释：________

**教师说明（不投影）**

我们把最低的一档叫零，最高的一档叫三。每个编号固定占几位，才能区分四种状态？两位。请写满两位，不要把零写成一位。写成码字这一步叫编码。这里约定的是课堂等级编号，真实PCM还会约定整数的表示和字节顺序。

追问：把编号3保存为“11”，这个动作有没有再次决定采样时刻？

技术结论：采样取值，量化近似，编码写码字；本课编码不等于压缩。

## Q7 只有码字，就能按原速播放吗？

00 01 10 11；同一张四等级码表。
4个样本，能确定这段声音的时长吗？

预测：________

观察后解释：________

**教师说明（不投影）**

码表已经给了，编号也读得懂，可是每隔多久读一个？如果每秒四个，这四个样本对应一秒；每秒八个，就对应半秒。数据要连着解释规则一起保存。后面正常的采样率对照，会同时正确记录新采样率，播放时长保持六秒，不是给音乐加速。

追问：把低采样率文件正确播放，与把同一批样本用更高的播放采样率播放，一样吗？

技术结论：样本序列与解释参数共同确定数字音频。

## Q8 只增加采样率，声音会怎样？

同一段6秒音乐，单声道、16bit。
比较8kHz与44.1kHz，其他条件不变。
再看：8kHz能保留6kHz纯音吗？

预测：________

观察后解释：________

**教师说明（不投影）**

现在比较采样率高低，位数、声道、时长不动。先猜音乐会不会加速，再听，仍是六秒。听感是这组例子的观察，不能据此推出所有声音的规律。这里给出本课使用的技术条件：带限信号要保留到最高频率，采样率须高于它的两倍；实际还要给滤波留余量。六千赫兹需要高于一万二的采样率，八千不够，所以正常降采样先把这段高频滤掉。若基准纯音也听不到，我们直接用条件判断，不拿静音当证明。

追问：8kHz的半数是多少？6kHz与它相比怎样？

技术结论：带限条件下理想重建要求`fs > 2fmax`；实际留滤波余量。采样率影响频率保留范围，不能只解释为折线更像原线。

### D2 先预测，后运行

In [ ]:
show_audio('music-44100-16-mono.wav')
show_audio('music-8000-16-mono.wav')

**等待讨论。下面单元只在完成预测后运行。**

In [ ]:
show_audio('tone-6000-at24000.wav')
show_audio('tone-lowpass-at8000.wav')

## Q9 采样率越高，听起来一定越好吗？

一段信号只含8kHz以下的频率。
32kHz已满足条件，改成96kHz一定更好听吗？

预测：________

观察后解释：________

**教师说明（不投影）**

这次题目给了一个重要条件：信号只有八千赫兹以下的成分，三万二已经足够。改成九万六会让数更多，但不能保证耳朵觉得更好。我们用的M4A本身已是压缩文件，转成更大的PCM WAV也不能恢复之前损失的信息。参数要和需求一起判断。

追问：源文件已经丢掉的细节，单纯转换成更高采样率会自己回来吗？

技术结论：音质关系应带条件；更高采样率不保证可闻改进。

## Q10 采样时刻不变，怎样减小量化误差？

同一批样本，幅度范围固定。
2bit与4bit量化，哪一种近似更细？

预测：________

观察后解释：________

**教师说明（不投影）**

这次所有取样时刻都不动，只增加可选等级。每个样本与近似值之间的红线表示误差。四位时可以分十六档，误差范围缩小。再听同一段音乐：四位有效模型常能听出粗糙感。没有听出来也先看图。这里播放器装的是十六位WAV；四位说的是我们模拟的量化等级，稍后算文件大小要读真实存储位数。

追问：红线是在横向补时刻，还是纵向修近似？

技术结论：固定范围、均匀量化下位数增加会减小步长及最大近似误差；不声称每个特定样本的误差都严格单调下降。

### D3 先预测，后运行

In [ ]:
plot_quantization(2, False)

**等待讨论。下面单元只在完成预测后运行。**

In [ ]:
plot_quantization(2, True)
plot_quantization(4, True)
show_audio('music-22050-effective4-stored16-mono.wav')
show_audio('music-22050-16-mono.wav')

## Q11 量化位数加倍，等级也只加倍吗？

2bit、4bit、8bit各有多少个码字？
16bit最多能区分多少个等级？

预测：________

观察后解释：________

**教师说明（不投影）**

两位不是两档，四位也不是四档。每一位有零和一两种状态，组合起来是二的n次方。四位相对两位，位数加倍，档数从四变成十六。把“每样本占几位”与“可选多少等级”分开写。

追问：每增加1bit，码字数怎样变化？

技术结论：等级数由`2ⁿ`确定；位数也是每样本的存储成本之一。

## Q12 三步分别改变了什么？

采样、量化、编码，分别回答哪一个问题？
什么时候读？幅度分多细？怎样写码字？

预测：________

观察后解释：________

**教师说明（不投影）**

现在先不背定义，用刚才做过的动作说。挑时刻是采样，挑代表等级是量化，写码字是编码。采样和量化各自限制一种精度。把码字写得再长，也不会凭空出现没记录的频率内容。

追问：增加位数能补回没有记录的快速变化吗？

技术结论：用时间、幅度、表示规则构成音频数字化模型。

## Q13 左右两路声音，怎样记录？

同采样率、同位数、同样6秒。
单声道1路，双声道2路；样本数据量怎样变？

预测：________

观察后解释：________

**教师说明（不投影）**

这里的声道是一路信号。每个时刻，左边一个数，右边一个数，就有两个样本。我们只是数存储量，不能由两倍数据推出两倍音量或两倍音质。图像RGB通道是颜色分量，音频L、R是声音的两路信号，别把含义混在一起。

追问：把相同的单声道复制到两路，信息内容一定增加了吗？

技术结论：声道数是样本计数的乘数。

## Q14 未压缩PCM WAV保存了什么？

要让别人读懂并播放这些样本，
文件除了样本，还需要记录什么？

预测：________

观察后解释：________

**教师说明（不投影）**

采样率告诉我们多久读一个，位数告诉我们怎么分组，声道告诉我们怎样分成各路。WAV把这些格式信息和样本装在一起。今天只讨论未压缩整数PCM WAV：它按固定存储位数保存样本，没有再做AAC那样的音频压缩。不过WAV这个容器也能装其他格式，不能只看扩展名下结论。

追问：PCM编码和MP3／AAC压缩是一回事吗？所有WAV都一定未压缩吗？

技术结论：未压缩PCM可按样本数与实际存储位数计算；WAV是容器。

## Q15 怎样从样本数推导数据量？

每秒每声道fs个样本，录音t秒。
每样本n bit，共C个声道。
先数样本，再换算成B。

预测：________

观察后解释：________

**教师说明（不投影）**

先别急着背。每秒一条路有fs个样本，t秒是多少？再有C条路，一共多少个数？每个数占n位，现在得到的是bit。要用B表示，还要除以八。每一个乘数都能在记录动作里找到。这里算的是样本数据，不包含文件头。

追问：式子的哪个因子把bit换成了byte？哪个因子记录了时间？

技术结论：从样本计数推导公式，适用范围为固定存储位数的未压缩PCM。

## Q16 一分钟双声道PCM有多大？

44.1kHz、16bit、双声道，录音1分钟。
求样本数据量（B）；换算MiB列式即可。
1 MiB = 1,048,576 B。

预测：________

观察后解释：________

**教师说明（不投影）**

请各自写，不先看答案。先把四十四点一千赫兹换成赫兹，把一分钟换成秒。双声道乘二，bit换B除八。先求B，再按给定的1MiB换算；列出分式就可以，我来给近似值。MiB用1024进位，MB是十进制单位，算式与单位必须配套。

追问：如果采样率不变、位数不变，只把时长变为30秒，怎样快速判断？

技术结论：同条件时数据量随时长、采样率、存储位数、声道数线性变化。

## Q17 算出32000B，文件一定也是32000B吗？

实际样例：8kHz、2秒、16bit、单声道PCM WAV。
公式得到32,000B。完整文件会一样大吗？

预测：________

观察后解释：________

**教师说明（不投影）**

你算出三万二，文件却是三万二千零四十四。分开数：样本一万六千个，每个两个byte，正好三万二；多出的结构信息只属于这个样例，不能说所有WAV都是四十四字节头。再回看开场A/B：都六秒、16bit、单声道，采样率却分别是44.1k和8k，样本数据因此是529,200B和96,000B。请核对我们最初两条猜测：哪条被实际参数支持，哪条要修改？这次大小差不是压缩程度造成的。

追问：如果WAV含额外说明、其他chunk或更复杂头部，开销一定相同吗？

技术结论：真实计数验证公式的payload边界；已知参数时能解释同长未压缩音频的大小差异。开场冲突到此得到证据支持的解释。

### D4 先预测，后运行

In [ ]:
# 先预测公式结果与完整文件大小是否相同。

**等待讨论。下面单元只在完成预测后运行。**

In [ ]:
info = pcm_info(ROOT / 'assets/audio/size-check-8000-2s-16-mono.wav')
evidence = show_pcm_evidence('size')

先解释公式结果与完整文件的差额，再运行下面单元核对开场猜测。

In [ ]:
opening_evidence = show_pcm_evidence('opening')

## Q18 只有64KiB，哪种方案满足要求？

带限信号最高频率6kHz；记录2秒。
16bit、单声道；上限64KiB = 65,536B（仅样本数据）。
候选采样率：8k、16k、24kHz。逐项检查两个条件。

预测：________

观察后解释：________

**教师说明（不投影）**

题目给定信号最高到六千赫兹，按刚才的理想条件，采样率要高于一万二。再看记录两秒、16bit、单声道的数据量，有没有超过给定的65,536B？先各自检查一个方案，再同桌合并。最后不只报一万六，请分别指出八千和二万四输在哪一关。这里是在选参数，没有试听一段不存在的16kHz证据文件。

追问：64KiB到底是多少B？选较高参数前先检查哪个限制？

技术结论：按需求和预算作条件判断，说明取舍。

## Q19 哪些说法需要修改？

① 8kHz采样，声音自身频率就是8kHz。
② 16bit量化只有16个等级。
③ 所有WAV都未压缩，公式算出完整文件大小。

预测：________

观察后解释：________

**教师说明（不投影）**

最后每个人选一句改写，并在旁边写一条依据。先给大家四十五秒。谁愿意读自己的改写？第三句有两个错误：WAV不一定未压缩，公式也不包含文件头；刚才的三万二和三万二千零四十四只直接反驳第二个错误。采样、量化、编码分别解决时刻、有限幅度和表示规则，再用样本计数检查数据量。

追问：哪一条可以用今天的实际文件字节数直接反驳？

技术结论：音频数字化是采样、量化、编码；数据量计算有明确存储格式边界。

## 离线实验台备用
需要逐个读点动画时使用；嵌入加载失败则用PPTX静态图与预制WAV。

In [ ]:
display(IFrame(src='demos/audio-lab.html', width='100%', height=720))

## 课后练习

1. 22.05kHz、16bit、单声道10秒，样本数据量是多少B？
2. 4bit变成8bit，等级数变成几倍？
3. 把8kHz录音转换成48kHz，能恢复未记录的6kHz吗？

教师答案：441,000B；16倍；不能恢复未记录或已丢失的信息。